## NB02 — Threshold calibration (fixed-FPR protocol)

**Provenance.** All logic in this notebook is copied verbatim from
`src/calibrate_fpr.py` in the project repository
(`github.com/sir-iko/id-forgery-detection`). Nothing is loaded from a
pre-computed file. Every number below is recomputed live from the three frozen
per-sample score files and is deterministic:

- Inputs: `checkpoints/scores_test_{resnet50,densenet121,vit}.csv`
  (columns `idx, y_true, p_forged, attack_type`; `p_forged` stored to 6 dp).
- The calibration partition is fixed by `PARTITION_SEED = 42`.
- Each repeat draw is fixed by `seed = 1000 * size + r`, so the sweep reproduces
  draw for draw with no global RNG state.

**What this notebook tests.** Whether either baseline failure (the recoverable
face-swap signal or the inverted text-edit ranking established in NB01) can be
repaired by choosing an operational threshold, rather than by changing the
model. It does not retrain anything. It fits one decision threshold on genuine
documents and measures what that threshold recovers per attack type.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, matthews_corrcoef,
    roc_auc_score,
)

# Frozen per-sample score files (one row per test image).
SCORE_FILES = {
    "ResNet50":    "checkpoints/scores_test_resnet50.csv",
    "DenseNet121": "checkpoints/scores_test_densenet121.csv",
    "ViT-B/16":    "checkpoints/scores_test_vit.csv",
}

# Protocol constants (identical to src/calibrate_fpr.py).
PARTITION_SEED = 42
PARTITION_SIZE = 150
CALIB_SIZES    = [10, 25, 50]
N_REPEATS      = 20
TARGET_FPR     = 0.10

STAGES = ["global", "face", "text"]

### Calibration logic (inlined verbatim from `src/calibrate_fpr.py`)

The functions below are the exact procedure that produced the reported
calibration numbers. `stratified_draw` uses a largest-remainder allocation so a
draw of size *n* preserves the attack-type composition of the partition.
`fit_threshold_fixed_fpr` fits **one** threshold on the bonafide rows of a draw:
the `(1 - target_fpr)` empirical quantile of bonafide `p_forged`, taken with
`method="higher"` so the realised false-positive rate does not exceed the
target. That single threshold is then applied to every reporting subset. This
mirrors deployment, where one operating point is set on genuine documents and
held across attack types.

In [2]:
def load_scores(csv_path):
    """Read a frozen scores CSV (cols: idx, y_true, p_forged, attack_type)."""
    df = pd.read_csv(csv_path)
    expected = {"idx", "y_true", "p_forged", "attack_type"}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path}: missing columns {missing}")
    return df


def stratified_draw(df, n, seed, key="attack_type"):
    """Draw n rows from df, stratified by `key`, with a fixed seed.
    Identical to calibrate.py so draws match draw-for-draw."""
    rng = np.random.default_rng(seed)
    groups = {k: sub.index.to_numpy() for k, sub in df.groupby(key)}
    total = len(df)
    raw = {k: n * len(idx) / total for k, idx in groups.items()}
    floor = {k: int(np.floor(v)) for k, v in raw.items()}
    remainder = n - sum(floor.values())
    frac_order = sorted(raw, key=lambda k: raw[k] - floor[k], reverse=True)
    for k in frac_order[:remainder]:
        floor[k] += 1
    picked = []
    for k, idx in groups.items():
        take = min(floor[k], len(idx))
        picked.extend(rng.choice(idx, size=take, replace=False).tolist())
    return df.loc[picked]


def carve_partition(df):
    """Carve the frozen calibration partition once, stratified. Identical to
    calibrate.py, so the partition and reporting set match exactly."""
    partition = stratified_draw(df, PARTITION_SIZE, PARTITION_SEED)
    reporting = df.drop(index=partition.index)
    return partition, reporting


def fit_threshold_fixed_fpr(draw, target_fpr=TARGET_FPR):
    """One GLOBAL threshold fit on the bonafide rows of a calibration draw.
    Bonafide are attack_type == 'none' (y_true == 0). The threshold is the
    lowest score t such that at most target_fpr of bonafide are flagged
    (p_forged >= t). Concretely: the (1 - target_fpr) empirical quantile of
    the bonafide scores, using 'higher' interpolation so the realised FPR
    does not exceed the target. Returns the threshold, or None if the draw
    has no bonafide (cannot define FPR)."""
    bona = draw[draw["attack_type"] == "none"]["p_forged"].to_numpy()
    if len(bona) == 0:
        return None
    # (1 - fpr) quantile of bonafide scores; 'higher' keeps realised FPR <= target
    t = float(np.quantile(bona, 1.0 - target_fpr, method="higher"))
    return t


def apply_threshold(p_forged, t):
    return (p_forged >= t).astype(int)


def recovery_metrics(y_true, y_pred):
    """Same metric set as calibrate.py: balanced_accuracy and mcc are the
    threshold-fair primaries; f1 is the cautionary secondary."""
    return {
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
    }


def static_auc(y_true, p_forged):
    if len(np.unique(y_true)) < 2:
        return None
    return float(roc_auc_score(y_true, p_forged))


def subset_by_type(df, atk):
    """Reporting subset for one attack type: that type's rows plus all
    bonafide rows. Identical to calibrate.py."""
    if atk == "none":
        return df
    return df[(df["attack_type"] == atk) | (df["attack_type"] == "none")]


def run_recovery_fixed_fpr(calib_partition, reporting, stages, target_fpr=TARGET_FPR):
    """Sweep calibration-set sizes. For each size and repeat, draw ONCE from
    the full partition, fit ONE fixed-FPR threshold on that draw's bonafide,
    then apply it to EVERY stage's reporting subset."""
    per_stage = {}
    for atk in stages:
        rp = reporting if atk == "global" else subset_by_type(reporting, atk)
        per_stage[atk] = {
            "static_auc": static_auc(rp["y_true"].to_numpy(),
                                     rp["p_forged"].to_numpy()),
            "n_report": int(len(rp)),
            "sizes": {},
        }
    report_bona = reporting[reporting["attack_type"] == "none"]["p_forged"].to_numpy()

    for size in CALIB_SIZES:
        if size > len(calib_partition):
            continue
        stage_runs = {atk: [] for atk in stages}
        realised_fpr = []
        n_thresh = 0
        for r in range(N_REPEATS):
            draw = stratified_draw(calib_partition, size, seed=1000 * size + r)
            t = fit_threshold_fixed_fpr(draw, target_fpr=target_fpr)
            if t is None:
                continue
            n_thresh += 1
            if len(report_bona) > 0:
                realised_fpr.append(float((report_bona >= t).mean()))
            for atk in stages:
                rp = reporting if atk == "global" else subset_by_type(reporting, atk)
                pred = apply_threshold(rp["p_forged"].to_numpy(), t)
                stage_runs[atk].append(
                    recovery_metrics(rp["y_true"].to_numpy(), pred))
        for atk in stages:
            runs = stage_runs[atk]
            if not runs:
                continue
            agg = {}
            for m in ("balanced_accuracy", "mcc", "f1"):
                vals = np.array([x[m] for x in runs])
                agg[m] = {"mean": float(vals.mean()),
                          "std": float(vals.std())}
            per_stage[atk]["sizes"][size] = agg
        if realised_fpr:
            per_stage["global"].setdefault("realised_fpr", {})[size] = {
                "mean": float(np.mean(realised_fpr)),
                "std": float(np.std(realised_fpr)),
                "n_valid_draws": n_thresh,
            }
    return per_stage

### Run the sweep

Each model is scored independently. The partition is carved once per model
(seed 42), the sweep runs over calibration sizes 10, 25 and 50 with 20 repeats
each, and metrics are averaged over the repeats. `static_auc` is the
threshold-independent AUC on each reporting subset and is printed alongside for
reference.

In [3]:
results = {}
for name, path in SCORE_FILES.items():
    df = load_scores(path)
    partition, reporting = carve_partition(df)
    results[name] = run_recovery_fixed_fpr(partition, reporting, STAGES)
    n_bona = int((partition["attack_type"] == "none").sum())
    print(f"{name:12s}  partition={len(partition)} (bonafide {n_bona})  "
          f"reporting={len(reporting)}")

ResNet50      partition=150 (bonafide 33)  reporting=1235


DenseNet121   partition=150 (bonafide 33)  reporting=1235


ViT-B/16      partition=150 (bonafide 33)  reporting=1235


### Recovery table (balanced accuracy and MCC, mean ± std over 20 draws)

Balanced accuracy is the primary recovery metric because the per-attack
reporting subsets are imbalanced. MCC is reported alongside because it exposes
inverted predictions that a balanced accuracy near 0.5 can mask: a negative MCC
means the thresholded classifier is anti-correlated with truth, wrong more often
than right. Values are shown to three decimal places to match the report.

In [4]:
def fmt(d, m):
    return f"{d[m]['mean']:.3f} \u00b1 {d[m]['std']:.3f}"

rows = []
for name in SCORE_FILES:
    ps = results[name]
    for atk in ["face", "text", "global"]:
        for size in CALIB_SIZES:
            if size not in ps[atk]["sizes"]:
                continue
            d = ps[atk]["sizes"][size]
            rows.append({
                "Model": name,
                "Stage": atk,
                "Calib size": size,
                "Balanced acc": fmt(d, "balanced_accuracy"),
                "MCC": fmt(d, "mcc"),
            })

table = pd.DataFrame(rows).set_index(["Model", "Stage", "Calib size"])
table

Balanced acc             MCC
Model       Stage  Calib size                               
ResNet50    face   10          0.765 ± 0.052   0.562 ± 0.087
                   25          0.789 ± 0.025   0.644 ± 0.039
                   50          0.796 ± 0.007   0.631 ± 0.042
            text   10          0.407 ± 0.089  -0.239 ± 0.159
                   25          0.480 ± 0.018  -0.113 ± 0.053
                   50          0.467 ± 0.024  -0.149 ± 0.058
            global 10          0.457 ± 0.081  -0.065 ± 0.156
                   25          0.523 ± 0.015   0.074 ± 0.045
                   50          0.513 ± 0.021   0.042 ± 0.058
DenseNet121 face   10          0.765 ± 0.046   0.555 ± 0.074
                   25          0.779 ± 0.028   0.589 ± 0.027
                   50          0.788 ± 0.018   0.600 ± 0.018
            text   10          0.427 ± 0.070  -0.216 ± 0.109
                   25          0.460 ± 0.024  -0.167 ± 0.045
                   50          0.461 ± 0.008  -0.169 ± 0.015
            global 10          0.474 ± 0.064  -0.040 ± 0.118
                   25          0.504 ± 0.019   0.016 ± 0.043
                   50          0.507 ± 0.005   0.018 ± 0.013
ViT-B/16    face   10          0.643 ± 0.061   0.329 ± 0.084
                   25          0.635 ± 0.054   0.336 ± 0.053
                   50          0.681 ± 0.042   0.388 ± 0.049
            text   10          0.401 ± 0.082  -0.241 ± 0.115
                   25          0.452 ± 0.050  -0.166 ± 0.086
                   50          0.441 ± 0.024  -0.203 ± 0.037
            global 10          0.435 ± 0.072  -0.120 ± 0.119
                   25          0.477 ± 0.038  -0.045 ± 0.073
                   50          0.474 ± 0.015  -0.063 ± 0.028

### Threshold-independent reference: static AUC and realised FPR

The static AUC per reporting subset and the realised bonafide FPR at each
calibration size are printed below. The realised FPR shows how close the fitted
operating point lands to the 10% target. At size 10 the partition supplies only
two or three bonafide images per draw, so the cut is estimated from a fraction
of an expected false positive and is necessarily coarse; the std over 20 repeats
exposes this instability directly. This small-sample instability is itself a
finding about how few labelled images a fixed-FPR calibration can be trusted
with.

In [5]:
for name in SCORE_FILES:
    ps = results[name]
    print(f"=== {name} ===")
    for atk in ["face", "text", "global"]:
        auc = ps[atk]["static_auc"]
        auc_s = f"{auc:.4f}" if auc is not None else "n/a"
        print(f"  {atk:8s}  static AUC={auc_s}  (n_report={ps[atk]['n_report']})")
    rf = ps["global"].get("realised_fpr", {})
    for size in CALIB_SIZES:
        if size in rf:
            print(f"    realised FPR @ size {size:3d}: "
                  f"{rf[size]['mean']:.3f} +/- {rf[size]['std']:.3f}")
    print()

=== ResNet50 ===
  face      static AUC=0.9207  (n_report=401)
  text      static AUC=0.1440  (n_report=1101)
  global    static AUC=0.2515  (n_report=1235)
    realised FPR @ size  10: 0.235 +/- 0.223
    realised FPR @ size  25: 0.049 +/- 0.045
    realised FPR @ size  50: 0.082 +/- 0.060

=== DenseNet121 ===
  face      static AUC=0.8921  (n_report=401)
  text      static AUC=0.1625  (n_report=1101)
  global    static AUC=0.2635  (n_report=1235)
    realised FPR @ size  10: 0.190 +/- 0.188
    realised FPR @ size  25: 0.103 +/- 0.065
    realised FPR @ size  50: 0.099 +/- 0.022

=== ViT-B/16 ===
  face      static AUC=0.7835  (n_report=401)
  text      static AUC=0.1966  (n_report=1101)
  global    static AUC=0.2778  (n_report=1235)
    realised FPR @ size  10: 0.300 +/- 0.285
    realised FPR @ size  25: 0.131 +/- 0.143
    realised FPR @ size  50: 0.155 +/- 0.071



### Findings

**Face recovers.** For all three architectures the face stage rises well above
chance with a positive MCC. Recovery generally strengthens as the calibration
size grows, but not monotonically: ViT-B/16 balanced accuracy dips at size 25
(0.643 to 0.635 to 0.681 across sizes 10, 25, 50), and the ResNet50 face MCC
peaks at size 25 rather than 50. The face failure identified in NB01 is
therefore a thresholding problem, repairable with a modest labelled partition.

**Text stays inverted.** For all three architectures the text stage sits below
0.5 balanced accuracy at every calibration size, with a negative MCC throughout.
No calibration size moves the text stage out of the inverted regime. A threshold
cannot recover a ranking that is already inverted. This is the operational
confirmation that the text failure is representational, not a matter of where
the decision boundary sits.

**Pooled collapses toward chance.** The global stage, which applies one
threshold across a reporting pool dominated by text-edit forgeries, sits near
chance with an MCC close to zero. Pooling averages the recoverable face signal
against the inverted text signal and the two cancel. This reproduces the pooled
evaluation regime on the present data and shows why per-attack decomposition is
necessary to see the two failure modes at all.

The claim these results support is specific to the model class under test:
off-the-shelf, whole-image, ImageNet-pretrained classifiers invert on text
manipulation and that inversion survives threshold calibration. It is not a
claim that text manipulation is universally undetectable.

### Note on subset sizes (consistency with NB01)

The static AUCs above are computed on the calibration **reporting** subsets,
which exclude the 150-image calibration partition carved out by seed 42. The
reporting face subset therefore holds fewer images than the whole-test face pool
used in NB01, and likewise for text. The AUCs here consequently differ in the
low decimal places from the NB01 baseline table. This is a difference of
denominator, not a disagreement: NB01 reports whole-test per-attack AUC, this
notebook reports AUC on the post-partition reporting pool. Both are read from
the same frozen score files.